In [1]:
import pickle
import tensorflow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import GlobalMaxPooling2D
from numpy.linalg import norm
import numpy as np
import os
from tqdm import tqdm

# Create the base ResNet50 model
model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model.trainable = False

# Wrap in Sequential with GlobalMaxPooling2D
model = tensorflow.keras.Sequential([
    model,
    GlobalMaxPooling2D()
])

def extract_features(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    expanded_img_array = np.expand_dims(img_array, axis=0)  # Expand dims to match model input
    preprocessing_img = preprocess_input(expanded_img_array)  # Correct preprocessing
    result = model.predict(preprocessing_img).flatten()
    normalized_result = result / norm(result)  # Normalize feature vector
    return normalized_result

# Collect image filenames
filenames = [os.path.join('images', file) for file in os.listdir('images')]

# Extract features
feature_list = [extract_features(file, model) for file in tqdm(filenames)]

# Save extracted features and filenames
pickle.dump(feature_list, open('embeddings.pkl', 'wb'))
pickle.dump(filenames, open('filenames.pkl', 'wb'))

# from PIL import Image
# print("Pillow is installed and working!")

100%|██████████| 39816/39816 [39:02<00:00, 17.00it/s]  
